## Analysis of Results

#### Setup

In [2]:
### Imports ###
import os
import shutil
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor, as_completed
import matplotlib.lines as mlines
import numpy as np
from matplotlib.patches import Patch
from pathlib import Path

In [3]:
work_dir = "/jumbo/keller-lab/Jeremy_Wang/eplus_sa/scripts/main" # Change this to your working directory
os.chdir(work_dir) # Set working directory

In [4]:
# --- Load Simulation Results for all seeds ---

# number of sims per seed
num_simulations = 30000

# list your seed directories
seeds = [f"seed_{i}" for i in range(1,6)]

# where your outputs live
output_base = os.path.join(work_dir, "output_sobol")

def read_csv_for_sim(task):
    seed, sim_id = task
    folder   = f"randomized_{sim_id}" # sim id is number run within the seed
    csv_file = os.path.join(output_base, seed, folder, "eplusmtr.csv")
    # skip if missing or zero‐byte
    if not os.path.isfile(csv_file) or os.path.getsize(csv_file) == 0:
        return None
    try:
        df = pd.read_csv(csv_file)
    except pd.errors.EmptyDataError:
        return None
    df["Simulation_ID"] = sim_id
    df["seed"]          = seed
    return df

# build the full list of (seed, sim_id) pairs
tasks = [(seed, i) for seed in seeds for i in range(1, num_simulations+1)]

all_dfs = []
max_workers = os.cpu_count() or 4 # returns the number of available cpu cores
print(f"Reading {len(tasks)} files across {len(seeds)} seeds using {max_workers} workers...")

with ProcessPoolExecutor(max_workers=max_workers) as executor:
    future_to_task = {executor.submit(read_csv_for_sim, t): t for t in tasks} # for every task (seed, i) combo, tells the executor to read csv
    for future in as_completed(future_to_task): # as soon as completed, append df into a bigger output
        df = future.result()
        seed, sim_id = future_to_task[future]
        if df is not None:
            all_dfs.append(df)
        else:
            # you can comment this out if it's too noisy
            print(f"Skipping empty/missing file: {seed}/randomized_{sim_id}/eplusmtr.csv")

if not all_dfs:
    raise RuntimeError("No simulation CSV files found. Check your output directories.")

# combine into one big DataFrame
# # rows = ((num_simulations * num_seeds) - (missing files)) * 12 months
combined_df = pd.concat(all_dfs, ignore_index=True)

Reading 150000 files across 5 seeds using 64 workers...
Skipping empty/missing file: seed_1/randomized_3214/eplusmtr.csv
Skipping empty/missing file: seed_3/randomized_744/eplusmtr.csv
Skipping empty/missing file: seed_3/randomized_7061/eplusmtr.csv
Skipping empty/missing file: seed_5/randomized_3899/eplusmtr.csv
Skipping empty/missing file: seed_5/randomized_3906/eplusmtr.csv
Skipping empty/missing file: seed_5/randomized_3908/eplusmtr.csv
Skipping empty/missing file: seed_5/randomized_4294/eplusmtr.csv


In [10]:
analysis_sim_dir = os.path.join(work_dir, "analysis_sobol")
os.makedirs(analysis_sim_dir, exist_ok=True)

# --- Clean analysis output directory ---
for fn in os.listdir(analysis_sim_dir):
    path = os.path.join(analysis_sim_dir, fn)
    if os.path.isfile(path) or os.path.islink(path): 
        os.unlink(path)
    elif os.path.isdir(path):
        shutil.rmtree(path)

# --- CSV output directory ---
csv_out = os.path.join(analysis_sim_dir, "output_csv")
if os.path.isdir(csv_out): shutil.rmtree(csv_out) # removing the folder if it already exists
os.makedirs(csv_out, exist_ok=True)

In [19]:
### Convert numbers into numerics ###

# Convert columns to numeric
numeric_cols = [
    "Electricity:Facility [J](Monthly)", "Electricity:Building [J](Monthly)", "InteriorLights:Electricity [J](Monthly)",
    "Electricity:Facility [J](RunPeriod)", "Electricity:Building [J](RunPeriod)", "InteriorLights:Electricity [J](RunPeriod)",
    "Electricity:HVAC [J](Monthly)", "NaturalGas:Facility [J](Monthly)", "NaturalGas:HVAC [J](Monthly)",
    "Electricity:HVAC [J](RunPeriod)", "NaturalGas:Facility [J](RunPeriod)", "NaturalGas:HVAC [J](RunPeriod)"
]

for col in numeric_cols:
    if col in combined_df:
        combined_df[col] = pd.to_numeric(combined_df[col], errors="coerce")

# Unit conversions (adds columns in KJ, KWh, BTU)
for col in list(combined_df):
    if col.endswith("[J](Monthly)") or col.endswith("[J](RunPeriod)"):
        combined_df[col.replace("[J]", "[KJ]")] = combined_df[col] / 1e3
        combined_df[col.replace("[J]", "[kWh]")] = combined_df[col] / 3.6e6

BTU_conv = 0.000947817
for col in list(combined_df):
    if "[J](" in col and ("HVAC" in col or "NaturalGas" in col):
        combined_df[col.replace("[J]", "[BTU]")] = combined_df[col] * BTU_conv

# saving converted combined_df values to csv in the folder
combined_df.to_csv(os.path.join(csv_out, "combined_sims.csv"), index=False, float_format="%.2f")

# --- Define month order ---
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]

combined_df["Date/Time"] = pd.Categorical(
    combined_df["Date/Time"], categories=month_order, ordered=True
)

In [ ]:
### saving csv files by seed ###

# For a variable number of seeds (e.g., 1 to 20)
max_seed = 5  # Change this to the number of seeds you have

for i in range(1, max_seed + 1):
    seed_name = f"seed_{i}"
    # Check if this seed exists in the data before creating the variable
    if seed_name in combined_df["seed"].unique():
        globals()[seed_name] = combined_df[combined_df["seed"] == seed_name] # creates a df for each seed

for i in range(1, max_seed+1):
    seed_df = combined_df[combined_df["seed"] == f"seed_{i}"]
    seed_df.to_csv(
        os.path.join(csv_out, f"seed_{i}.csv"),
        index=False,
        float_format="%.2f"
    )

In [ ]:
### Calculate annual energy consumption ###
# note: the column electricity: facility [J] is total electricity consumption for the month
total_consumption_cols = ['Electricity:Facility [J](Monthly)', 'Electricity:Facility [KJ](Monthly)','Electricity:Facility [kWh](Monthly)','Simulation_ID', 'seed',]
total_consumption = combined_df[total_consumption_cols]

# summing up monthly consumption to annual consumption per simulation
annual_consumption = (
    total_consumption
    .groupby(['Simulation_ID','seed'], as_index=False)
    .agg(
        seed = ('seed', 'first'),
        **{
            col: (col, 'sum')
            for col in total_consumption_cols
            if col not in ['Simulation_ID', 'seed']
        }
    )
)

# order by seed and Simulation_ID
annual_consumption = annual_consumption.sort_values(
    by=['seed', 'Simulation_ID']
).reset_index(drop=True)

# renaming columns
annual_consumption = annual_consumption.rename(
    columns=lambda c: c.replace('(Monthly)', '(Annual)')
)

In [48]:
annual_consumption

,Simulation_ID,seed,Electricity:Facility [J](Annual),Electricity:Facility [KJ](Annual),Electricity:Facility [kWh](Annual)
0,1,seed_1,2.320335e+11,2.320335e+08,64453.737884
1,2,seed_1,2.529213e+11,2.529213e+08,70255.927635
2,3,seed_1,2.320757e+11,2.320757e+08,64465.466513
3,4,seed_1,2.321056e+11,2.321056e+08,64473.771281
4,5,seed_1,2.320339e+11,2.320339e+08,64453.874556
...,...,...,...,...,...
149988,29996,seed_5,1.779452e+11,1.779452e+08,49429.223570
149989,29997,seed_5,1.779462e+11,1.779462e+08,49429.495982
149990,29998,seed_5,1.779462e+11,1.779462e+08,49429.495982
149991,29999,seed_5,1.940167e+11,1.940167e+08,53893.517188
